# DubbingStory — Qwen3-VL Local Vision (Kaggle + Colab)

**Universal notebook:** otomatis mendeteksi Kaggle atau Google Colab, memuat secret dengan backend yang sesuai, lalu menjalankan pipeline yang sama.


## 1. Clone Repo

Download source code project dari branch `main`.


In [ ]:
# Clone repository langsung ke current directory agar tidak nested
REPO_URL = "https://github.com/NaufalRizqullah/dubbingstory.git"
GIT_BRANCH = "main"  # ganti ke branch patch, mis. "vision-architecture-fix", sebelum merge ke main

!rm -rf ./* ./.[!.]* ./..?* 2>/dev/null || true
!git clone -b {GIT_BRANCH} {REPO_URL} .


## 2. Konfigurasi Pipeline

Default terbaru memakai **Qwen3-VL-4B-Instruct**, normal scene budget 640 token, clean-truncation retry 896 token, dan runaway rescue terpisah.


In [ ]:
# --- SETTINGS ---
VIDEO_INPUT = "https://www.youtube.com/watch?v=ms4wRkLIO5U"  # Bisa URL atau path file lokal
RESOLUTION = "1080"
PROJECT_NAME = "my_dubbing_project"
STYLE = "viral_fb"
LANGUAGE = "id"
RATIO = "16:9"
ENGINE_TTS = "edge"

# --- PIPELINE MODE ---
MODE = "summary"               # "full" atau "summary"
SUMMARY_DURATION = 60 * 3      # Target durasi ringkasan (detik), None = otomatis
SUMMARY_MAX_SCENES = None      # Maks scene, None = otomatis

# --- SCENE / VISION INPUT SETTINGS ---
MAX_KEYFRAMES = 3              # Ambil maksimal 3 frame representatif untuk setiap scene/window agar vision tetap cepat dan tidak membebani context token.
MIN_SCENE_DURATION = 3.0       # Scene yang lebih pendek dari 3 detik akan digabung/diabaikan agar tidak terlalu banyak potongan kecil yang kurang bermakna.
MAX_SCENE_DURATION = 15.0      # Scene yang lebih panjang dari 15 detik akan dipecah menjadi beberapa window agar perubahan visual/aksi tidak dianalisis sebagai satu scene panjang.
SCENE_THRESHOLD = 4.0          # Sensitivitas deteksi pergantian scene; makin kecil = lebih sensitif/banyak scene, makin besar = lebih sedikit scene.

# --- VISION MODEL ---
# Default terbaru: 4B. Untuk GPU lebih kecil bisa kembali ke Qwen/Qwen3-VL-2B-Instruct.
MODEL_NAME = "Qwen/Qwen3-VL-4B-Instruct"
PORT = 8000
BASE_URL = f"http://127.0.0.1:{PORT}/v1"

# Normal scene budget. Jangan membesarkan ini sebagai obat runaway.
VISION_MAX_TOKENS = "640"

# Qwen3-VL sampling baseline (non-greedy).
# temperature=0 sebelumnya dapat membuat decoder lebih mudah masuk jalur repetitif.
VISION_TEMPERATURE = "0.7"
VISION_TOP_P = "0.8"
VISION_TOP_K = "20"
VISION_PRESENCE_PENALTY = "1.5"
VISION_REPETITION_PENALTY = "1.0"

# Token estimator:
# Implementasi terbaru menghitung visual tokens dari dimensi image.
# Nilai ini hanya fallback bila dimensi image tidak dapat dibaca.
VISION_IMAGE_TOKEN_COST = "512"

# finish_reason=length recovery:
# - clean truncation: frame tetap, budget output dinaikkan
# - runaway/repetition: 1 middle frame + compact rescue schema
# - temporal clean truncation: budget temporal lebih besar
VISION_TRUNCATION_RETRY_TOKENS = "896"
VISION_RUNAWAY_RESCUE_TOKENS = "448"
VISION_TEMPORAL_TRUNCATION_RETRY_TOKENS = "6144"
VISION_RUNAWAY_THRESHOLD = "0.35"

# vLLM deployment:
# - "auto": 2B/4B atau quantized -> data parallel bila muat per GPU
# - "data_parallel": 1 replica model per GPU
# - "tensor_parallel": satu model dibagi ke beberapa GPU
# - "single": satu GPU
VLLM_PARALLEL_MODE = "auto"
VLLM_QUANTIZATION = None
VLLM_VERSION = "0.27.1"
GPU_MEM_UTIL = "0.85"

# 4B default yang dipakai notebook terbaru.
# Jika kembali ke 2B dan VRAM cukup, 12288 juga bisa dipakai.
MAX_MODEL_LEN = "10240"
VISION_CONCURRENCY_OVERRIDE = None

# --- IMAGE MODE ---
IMAGE_MODE = "file"
LOCAL_MEDIA_ROOT = __import__("os").path.abspath(".")

# --- GPU TOPOLOGY / PARALLELISM RESOLVER ---
import re
try:
    import torch
    NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
    GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(NUM_GPUS)]
    GPU_VRAM_GB = [
        round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 1)
        for i in range(NUM_GPUS)
    ]
except Exception:
    NUM_GPUS = 0
    GPU_NAMES = []
    GPU_VRAM_GB = []

if NUM_GPUS < 1:
    raise RuntimeError("GPU tidak terdeteksi. Aktifkan GPU accelerator sebelum menjalankan vLLM.")

mode = VLLM_PARALLEL_MODE.strip().lower()
quantized = bool(VLLM_QUANTIZATION) or any(
    tag in MODEL_NAME.upper() for tag in ("AWQ", "GPTQ", "INT4", "W4A16", "4BIT")
)
match = re.search(r"-(\d+(?:\.\d+)?)B(?:-|$)", MODEL_NAME, flags=re.I)
params_b = float(match.group(1)) if match else None

if mode == "auto":
    if NUM_GPUS >= 2 and (quantized or (params_b is not None and params_b <= 4.5)):
        mode = "data_parallel"
    elif NUM_GPUS >= 2:
        mode = "tensor_parallel"
    else:
        mode = "single"

if mode == "data_parallel":
    DATA_PARALLEL_SIZE = NUM_GPUS
    TENSOR_PARALLEL_SIZE = 1
    VISION_CONCURRENCY = DATA_PARALLEL_SIZE
elif mode == "tensor_parallel":
    DATA_PARALLEL_SIZE = 1
    TENSOR_PARALLEL_SIZE = NUM_GPUS
    VISION_CONCURRENCY = min(2, max(1, NUM_GPUS))
elif mode == "single":
    DATA_PARALLEL_SIZE = 1
    TENSOR_PARALLEL_SIZE = 1
    VISION_CONCURRENCY = 1
else:
    raise ValueError(f"VLLM_PARALLEL_MODE tidak dikenal: {VLLM_PARALLEL_MODE}")

if VISION_CONCURRENCY_OVERRIDE is not None:
    VISION_CONCURRENCY = max(1, int(VISION_CONCURRENCY_OVERRIDE))

required_gpus = DATA_PARALLEL_SIZE * TENSOR_PARALLEL_SIZE
if required_gpus > NUM_GPUS:
    raise RuntimeError(
        f"Konfigurasi membutuhkan {required_gpus} GPU, tetapi hanya {NUM_GPUS} terdeteksi."
    )

print(f"🖥️ GPUs: {list(zip(GPU_NAMES, GPU_VRAM_GB))}")
print(f"🧠 Model: {MODEL_NAME} | params~{params_b or '?'}B | quantized={quantized}")
print(
    f"⚙️ vLLM mode={mode}: DP={DATA_PARALLEL_SIZE}, "
    f"TP={TENSOR_PARALLEL_SIZE}, vision_concurrency={VISION_CONCURRENCY}"
)
print(f"📏 max-model-len={MAX_MODEL_LEN}, gpu-memory-utilization={GPU_MEM_UTIL}")
print(
    "🎛️ sampling: "
    f"temperature={VISION_TEMPERATURE}, top_p={VISION_TOP_P}, "
    f"top_k={VISION_TOP_K}, presence_penalty={VISION_PRESENCE_PENALTY}, "
    f"repetition_penalty={VISION_REPETITION_PENALTY}"
)
print(
    "🛟 length recovery: "
    f"truncation={VISION_TRUNCATION_RETRY_TOKENS}, "
    f"runaway={VISION_RUNAWAY_RESCUE_TOKENS}, "
    f"temporal={VISION_TEMPORAL_TRUNCATION_RETRY_TOKENS}, "
    f"threshold={VISION_RUNAWAY_THRESHOLD}"
)


## 3. Download Video & Subtitles

Video diunduh lebih dulu sebelum instalasi dependensi berat. Cookie YouTube bersifat opsional dan dideteksi otomatis.


In [ ]:
# Optional cookies: satu cell yang sama untuk Kaggle maupun Colab.
import os
import shutil

USE_COOKIES = True
COOKIES_PATH = os.path.abspath("youtube_cookies.txt")

# Kaggle: dataset cookie yang biasa digunakan.
# Colab: upload youtube_cookies.txt ke working directory (/content) bila diperlukan.
cookie_candidates = [
    "/kaggle/input/datasets/muhammadnaufal/tiktok-secret-cookies/youtube_cookies.txt",
    "/content/youtube_cookies.txt",
    COOKIES_PATH,
]

source_cookie = next((p for p in cookie_candidates if os.path.isfile(p)), None)

if source_cookie:
    if os.path.realpath(source_cookie) != os.path.realpath(COOKIES_PATH):
        shutil.copy(source_cookie, COOKIES_PATH)
    print(f"✅ YouTube cookies aktif: {COOKIES_PATH}")
else:
    USE_COOKIES = False
    print("ℹ️ youtube_cookies.txt tidak ditemukan. Download akan dicoba tanpa cookies.")


In [ ]:
import subprocess
import time
import urllib.request
import json
import sys
import os

print("   - Install Deno (JS runtime) + ffmpeg untuk membantu yt-dlp")
try:
    subprocess.check_call(
        ["apt-get", "-qq", "update"],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
    subprocess.check_call(
        ["apt-get", "-qq", "install", "-y", "ffmpeg", "curl", "unzip"],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
    subprocess.check_call(
        ["mkdir", "-p", "/root/.deno/bin"],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
    subprocess.check_call(
        [
            "curl", "-L", "--retry", "5", "--retry-all-errors", "--connect-timeout", "20",
            "-o", "/tmp/deno.zip",
            "https://github.com/denoland/deno/releases/latest/download/deno-x86_64-unknown-linux-gnu.zip",
        ],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
    subprocess.check_call(
        ["unzip", "-o", "/tmp/deno.zip", "-d", "/root/.deno/bin"],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
    subprocess.check_call(
        ["chmod", "+x", "/root/.deno/bin/deno"],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
    os.environ["PATH"] += ":/root/.deno/bin"
    subprocess.check_call(
        ["deno", "--version"],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
    print("   ✅ Deno terinstall dan PATH updated")
except Exception as e:
    print(f"   ⚠️ Gagal install Deno + ffmpeg (opsional): {e}")

print("   - Upgrade yt-dlp ke versi terbaru")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "yt-dlp"],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)


In [ ]:
import sys
import os
sys.path.append(".")

from dubbingstory.ingest.youtube import download_video, download_subtitles

print("📥 Memulai proses download video...")
PROJECT_DIR = os.path.join("outputs", PROJECT_NAME)
os.makedirs(PROJECT_DIR, exist_ok=True)

if "http" in VIDEO_INPUT:
    try:
        download_kwargs = {
            "download_height": RESOLUTION,
        }
        if USE_COOKIES:
            download_kwargs["cookies"] = COOKIES_PATH

        video_path = download_video(
            VIDEO_INPUT,
            PROJECT_DIR,
            **download_kwargs,
        )

        # Subtitle opsional untuk konteks tambahan model.
        download_subtitles(VIDEO_INPUT, PROJECT_DIR)
        print("\n✅ Download SUKSES!")
    except Exception as e:
        print(f"\n❌ DOWNLOAD GAGAL: {e}")
        print("\nJANGAN lanjutkan ke cell berikutnya. Periksa URL/cookies lalu coba lagi.")
        raise
else:
    print("ℹ️ VIDEO_INPUT bukan URL. Dianggap sebagai file lokal.")


## 4. Install Dependencies & Environment Optimization

Setelah download video sukses, instal dependensi project dan vLLM.


In [ ]:
# Install dependencies utama project
!pip install -q -r requirements.txt openai

import subprocess
import sys
import os

print("🔧 Fixing environment...")
print("   - Uninstall torchaudio (vLLM tidak membutuhkannya untuk pipeline ini)")
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchaudio"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
    check=False,
)

# --- DETEKSI KEMAMPUAN GPU ---
try:
    import torch
    if torch.cuda.is_available():
        cc_major, cc_minor = torch.cuda.get_device_capability()
    else:
        cc_major, cc_minor = 0, 0
except Exception:
    cc_major, cc_minor = 0, 0

if cc_major > 0 and cc_major < 7:
    print(f"   - Detected older GPU (Compute Capability {cc_major}.{cc_minor}).")
    print("   - Install PyTorch 2.7.1 (cu126) untuk kompatibilitas GPU lama.")
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", "--force-reinstall",
            "torch==2.7.1", "torchvision==0.22.1", "torchaudio==2.7.1",
            "--index-url", "https://download.pytorch.org/whl/cu126",
        ],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
else:
    print(f"   - Detected modern GPU (Compute Capability {cc_major}.{cc_minor}).")
    print("   - Install torch + torchvision sinkron CUDA 13.0.")
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
            "torch==2.13.0", "torchvision==0.28.0",
            "--index-url", "https://download.pytorch.org/whl/cu130",
        ],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )

print(f"   - Install vLLM {VLLM_VERSION} (pinned untuk reproducibility)")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", f"vllm=={VLLM_VERSION}"],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)

print("✅ Done. Lanjut ke setup secret lalu jalankan pipeline.")


## 5. Setup API Key & Vision Runtime Config

Notebook otomatis mendeteksi **Kaggle** atau **Google Colab**. Secret `GOOGLE_API_KEY` dibaca dengan API platform yang sesuai; di environment lain akan fallback ke environment variable.


In [ ]:
from pathlib import Path
import os
import sys

def detect_notebook_platform() -> str:
    """Return 'kaggle', 'colab', or 'other' without requiring platform packages."""
    # Strong environment hints first.
    if (
        os.environ.get("KAGGLE_KERNEL_RUN_TYPE")
        or os.environ.get("KAGGLE_URL_BASE")
        or os.path.exists("/kaggle")
    ):
        return "kaggle"

    if (
        os.environ.get("COLAB_RELEASE_TAG")
        or os.environ.get("COLAB_BACKEND_VERSION")
        or "google.colab" in sys.modules
    ):
        return "colab"

    # Import probes for cases where env vars are absent.
    try:
        import kaggle_secrets  # noqa: F401
        return "kaggle"
    except Exception:
        pass

    try:
        import google.colab  # noqa: F401
        return "colab"
    except Exception:
        pass

    return "other"


def load_google_api_key() -> tuple[str, str]:
    """Load GOOGLE_API_KEY from the current notebook platform."""
    platform = detect_notebook_platform()
    errors = []

    if platform == "kaggle":
        try:
            from kaggle_secrets import UserSecretsClient
            key = UserSecretsClient().get_secret("GOOGLE_API_KEY") or ""
            if key:
                return key, "Kaggle Secrets"
            errors.append("Kaggle secret GOOGLE_API_KEY kosong.")
        except Exception as exc:
            errors.append(f"Kaggle Secrets gagal: {exc}")

    elif platform == "colab":
        try:
            from google.colab import userdata
            key = userdata.get("GOOGLE_API_KEY") or ""
            if key:
                return key, "Colab Secrets"
            errors.append("Colab secret GOOGLE_API_KEY kosong.")
        except Exception as exc:
            errors.append(f"Colab Secrets gagal: {exc}")

    # Useful fallback for local/Jupyter/CI and also as a recovery path.
    key = os.environ.get("GOOGLE_API_KEY", "").strip()
    if key:
        return key, "environment variable"

    details = " | ".join(errors) if errors else "Secret backend platform tidak tersedia."
    raise RuntimeError(
        "GOOGLE_API_KEY tidak ditemukan. "
        "Tambahkan secret bernama GOOGLE_API_KEY di Kaggle/Colab "
        "atau set environment variable GOOGLE_API_KEY. "
        f"Detail: {details}"
    )


NOTEBOOK_PLATFORM = detect_notebook_platform()
API_KEY_GEMINI, API_KEY_SOURCE = load_google_api_key()

print(f"🧭 Notebook platform: {NOTEBOOK_PLATFORM}")
print(f"🔐 GOOGLE_API_KEY loaded from: {API_KEY_SOURCE}")

env_text = f"""# Auto-generated from notebook secret/runtime settings
GOOGLE_API_KEY={API_KEY_GEMINI}
OPENAI_VISION_IMAGE_MODE={IMAGE_MODE}
OPENAI_VISION_MAX_TOKENS={VISION_MAX_TOKENS}
OPENAI_VISION_MODEL_MAX_CONTEXT={MAX_MODEL_LEN}
OPENAI_VISION_STRUCTURED_OUTPUTS=true
OPENAI_VISION_ALLOW_SCHEMA_FALLBACK=false

# Qwen3-VL non-greedy sampling baseline.
OPENAI_VISION_TEMPERATURE={VISION_TEMPERATURE}
OPENAI_VISION_TOP_P={VISION_TOP_P}
OPENAI_VISION_TOP_K={VISION_TOP_K}
OPENAI_VISION_PRESENCE_PENALTY={VISION_PRESENCE_PENALTY}
OPENAI_VISION_REPETITION_PENALTY={VISION_REPETITION_PENALTY}

# Dimension-aware estimator is primary; this is fallback only.
OPENAI_VISION_IMAGE_TOKEN_COST={VISION_IMAGE_TOKEN_COST}

# Error-specific finish_reason=length recovery.
OPENAI_VISION_TRUNCATION_RETRY_TOKENS={VISION_TRUNCATION_RETRY_TOKENS}
OPENAI_VISION_RUNAWAY_RESCUE_TOKENS={VISION_RUNAWAY_RESCUE_TOKENS}
OPENAI_VISION_TEMPORAL_TRUNCATION_RETRY_TOKENS={VISION_TEMPORAL_TRUNCATION_RETRY_TOKENS}
OPENAI_VISION_RUNAWAY_THRESHOLD={VISION_RUNAWAY_THRESHOLD}

VISION_CONCURRENCY={VISION_CONCURRENCY}
"""

Path(".env").write_text(env_text, encoding="utf-8")

# Mirror to current process so execution order does not depend on dotenv reload.
for _line in env_text.splitlines():
    if not _line or _line.lstrip().startswith("#") or "=" not in _line:
        continue
    _key, _value = _line.split("=", 1)
    os.environ[_key] = _value

print("✅ .env terbaru dibuat")
print("   - language-safe narration retry: repo preflight OK")
print("   - Qwen sampling: non-greedy")
print("   - finish_reason=length: error-specific recovery aktif")


## 6. Jalankan Pipeline

Server vLLM dijalankan sebagai subprocess, lalu output pipeline di-stream ke notebook dan disimpan ke `pipeline.log`.


In [ ]:
import subprocess
import time
import urllib.request
import json
import sys
import os
from dotenv import load_dotenv

load_dotenv()

def wait_for_server(url, timeout=600):
    print(f"\n⏳ Waiting for vLLM server at {url} (timeout: {timeout}s)...")
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            req = urllib.request.Request(f"{url}/models")
            with urllib.request.urlopen(req) as response:
                if response.status == 200:
                    data = json.loads(response.read().decode())
                    print(f"\n✅ vLLM Server ready. Models: {[m['id'] for m in data['data']]}")
                    return True
        except Exception:
            pass
        sys.stdout.write(".")
        sys.stdout.flush()
        time.sleep(5)

    print("\n❌ Timeout waiting for vLLM server.")
    return False

if not os.environ.get("GOOGLE_API_KEY"):
    raise RuntimeError("GOOGLE_API_KEY belum tersedia. Jalankan cell Setup API Key terlebih dahulu.")

print(f"🔎 Runtime vision config: platform={NOTEBOOK_PLATFORM}")
print(f"   model={MODEL_NAME} | max_tokens={VISION_MAX_TOKENS} | max_context={MAX_MODEL_LEN}")
print(
    f"   sampling: T={VISION_TEMPERATURE}, top_p={VISION_TOP_P}, top_k={VISION_TOP_K}, "
    f"presence={VISION_PRESENCE_PENALTY}, repetition={VISION_REPETITION_PENALTY}"
)
print(
    f"   recovery: truncation={VISION_TRUNCATION_RETRY_TOKENS}, "
    f"runaway={VISION_RUNAWAY_RESCUE_TOKENS}, "
    f"temporal={VISION_TEMPORAL_TRUNCATION_RETRY_TOKENS}"
)

print(f"🚀 Starting vLLM server with model: {MODEL_NAME}...")
vllm_cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_NAME,
    "--port", str(PORT),
    "--max-model-len", str(MAX_MODEL_LEN),
    "--tensor-parallel-size", str(TENSOR_PARALLEL_SIZE),
    "--gpu-memory-utilization", str(GPU_MEM_UTIL),
    "--dtype", "half",
    "--generation-config", "vllm",  # request settings tidak ditimpa generation_config model repo
    "--enforce-eager",
]

if DATA_PARALLEL_SIZE > 1:
    vllm_cmd.extend(["--data-parallel-size", str(DATA_PARALLEL_SIZE)])

if VLLM_QUANTIZATION:
    vllm_cmd.extend(["--quantization", str(VLLM_QUANTIZATION)])

if IMAGE_MODE == "file":
    vllm_cmd.extend(["--allowed-local-media-path", LOCAL_MEDIA_ROOT])

print(f"   Hardware plan: DP={DATA_PARALLEL_SIZE}, TP={TENSOR_PARALLEL_SIZE}, requests={VISION_CONCURRENCY}")
print(f"   Command: {' '.join(vllm_cmd)}")

vllm_log = open("vllm_server.log", "w")
vllm_process = subprocess.Popen(
    vllm_cmd,
    stdout=vllm_log,
    stderr=subprocess.STDOUT,
)

try:
    if not wait_for_server(BASE_URL):
        raise RuntimeError("vLLM gagal start. Periksa vllm_server.log.")

    print(f"\n🚀 Starting DubbingStory Pipeline (mode: {MODE})...")

    local_video_path = os.path.join("outputs", PROJECT_NAME, "source.mp4")
    if "http" in VIDEO_INPUT and os.path.exists(local_video_path):
        cmd_input = local_video_path
        print(f"Menggunakan video yang sudah didownload: {cmd_input}")
    else:
        cmd_input = VIDEO_INPUT

    cmd = [
        sys.executable, "-u", "main.py", "run",
        "--input", cmd_input,
        "--project", PROJECT_NAME,
        "--style", STYLE,
        "--lang", LANGUAGE,
        "--ratio", RATIO,
        "--mode", MODE,
        "--vision-provider", "openai",
        "--vision-model", MODEL_NAME,
        "--vision-base-url", BASE_URL,
        "--vision-max-tokens", str(VISION_MAX_TOKENS),
        "--vision-concurrency", str(VISION_CONCURRENCY),
        "--engine", ENGINE_TTS,
        "--max-keyframes", str(MAX_KEYFRAMES),
        "--min-scene-duration", str(MIN_SCENE_DURATION),
        "--max-scene-duration", str(MAX_SCENE_DURATION),
        "--scene-threshold", str(SCENE_THRESHOLD),
    ]

    if MODE == "summary":
        if SUMMARY_DURATION is not None:
            cmd.extend(["--summary-duration", str(SUMMARY_DURATION)])
        if SUMMARY_MAX_SCENES is not None:
            cmd.extend(["--summary-max-scenes", str(SUMMARY_MAX_SCENES)])

    if "http" in cmd_input:
        cmd[cmd.index("--input")] = "--url"
        cmd.append("--i-have-rights")

    print(f"Executing: {' '.join(cmd)}")

    pipeline_log_path = "pipeline.log"
    with open(pipeline_log_path, "w") as plog:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            plog.write(line)
        returncode = proc.wait()

    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, cmd)

    print(f"\n🎉 Pipeline completed successfully! (mode: {MODE})")
    print(f"📂 Output: outputs/{PROJECT_NAME}/")

except Exception as e:
    print(f"\n❌ An error occurred: {e}")

    for log_name in ("pipeline.log", "vllm_server.log"):
        if os.path.exists(log_name) and os.path.getsize(log_name) > 0:
            print(f"\n──── tail of {log_name} ────")
            with open(log_name, "r", errors="replace") as lf:
                print("".join(lf.readlines()[-80:]))
    raise

finally:
    print("\n🛑 Shutting down vLLM server...")
    vllm_process.terminate()
    try:
        vllm_process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        vllm_process.kill()
        vllm_process.wait()
    vllm_log.close()
